# Feature selection experience

This notebook explores the contribution of each feature to predict close price in both scenario, binary classification and regression

## 1. Set up

We first set up the environment for experiment. This step include

1. Import all dependencies
2. Declare global environment to config the experiment
3. Declare all support functions needed for the experiment

### 1.1 Import all dependencies

This experiment requires the following libraries to run
- torch
- sklearn
- numpy

In [1]:
from src.data import Dataloader
import pandas as pd
import os
from src.utils import seed_everything
import random
from tqdm import tqdm
from typing import Callable
import torch.nn.functional as F
from src.data import TimeSeriesDataset
import torch
from sklearn.metrics import roc_auc_score, precision_score
import warnings
from sklearn.exceptions import DataConversionWarning

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import copy

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score
import warnings


### 1.2 Declare global environment to config the experiment

This section below will set up constant needed for experiment

In [ ]:
# Clean data location
DATA_PATH = os.path.abspath('data/clean/')

# Date that splits between training data and validation data in each token network (Unix time in seconds)
VAL_START_DATE = int(pd.Timestamp('2023-06-10').timestamp())  

# Date that splits between validation data and test data in each token network (Unix time in seconds)
TEST_START_DATE = int(pd.Timestamp('2023-09-15').timestamp())

# Columns that are used as features for training
ALL_FEAT_COLUMN = ['Close','High','Low', 'Open','Volume','sentiment_score_mean','sentiment_polarity_mean','sentiment_subjectivity_mean','news_count', 'log_return', 'vol_7d','vol_30d','ma_7','ma_30','ma_ratio']

# Columns that are used as features for training but do NOT need to normalize
ALREADY_NORMALIZE_FEAT = ['sentiment_score_mean','sentiment_polarity_mean','sentiment_subjectivity_mean','news_count','ma_ratio','log_return']

# Label for binary classification task
BINARY_LABEL_COLUMN = 'price_increase'

# Label for regression task
REGRESSION_LABEL_COLUMN = 'next_close'

# Number of tokens used in training to demonstrate scaling laws
LOG_SCALING = [2,4,8,16,32]

# For reproducibility, inital random seed
INIT_SEED = 720

seed_everything(INIT_SEED)

### 1.3. Declare all support functions needed for the experiment

To conduct experiment, we define the following support functions

1. `normalize`: normalize continuous features for training
2. `binary_classfication_model_experience`: Given each token dataset, run all baseline for binary classification task
3. `regression_classfication_model_experience`: Given each token dataset, run all baseline for regression classification task
4. `drop_feature_experiment` : conduct experiment to study importance of features by dropping each of them one by one

In [ ]:
def normalize(train_data: TimeSeriesDataset, valid_data: TimeSeriesDataset, test_data: TimeSeriesDataset, normalize_y = False, exclude_cols = []):
    r"""
    Normalize each feature in training, validation and testing data with max and min of the feature from training data.

    Args:
    - `train_data` (TimeSeriesDataset): data split for training
    - `valid_data` (TimeSeriesDataset): data split for validation
    - `test_data` (TimeSeriesDataset): data split for testing
    - `normalize_y` (bool): whether to normalize input (True for regresison case and False for binary classification case)
    - `exclude_cols` (list): columns that don't need to normalize

    Return:
    - data splits for training, validation and testing with normalized features
    """
    n_cols = train_data.x.shape[1]
    include_cols = [i for i in range(n_cols) if i not in (exclude_cols or [])]

    max_values, _ = torch.max(train_data.x[:, include_cols], dim=0)
    min_values, _ = torch.min(train_data.x[:, include_cols], dim=0)

    denom = (max_values - min_values)
    denom[denom == 0] = 1  # avoid division by zero

    for data in [train_data, valid_data, test_data]:
        data.x[:, include_cols] = (data.x[:, include_cols] - min_values) / denom

    if normalize_y:
        max_y, _ = torch.max(train_data.y, dim=0)
        min_y, _ = torch.min(train_data.y, dim=0)
        for data in [train_data, valid_data, test_data]:
            data.y = (data.y - min_y) / (max_y - min_y)

    return train_data, valid_data, test_data

In [ ]:
def binary_classfication_model_experience(train_data: TimeSeriesDataset, test_data: TimeSeriesDataset):
    r"""
    Train and evaluate multiple binary classification baseline models on time series data.

    Each model is trained on the training split and evaluated on the test split using
    ROC-AUC score as the performance metric. C
    Args:
    - `train_data` (TimeSeriesDataset): data split for training
    - `test_data` (TimeSeriesDataset): data split for testing

    Return:
    - `results` (dict): dictionary mapping model name to its ROC-AUC score on the test split
    """
    train_x = train_data.x.numpy()
    train_y = train_data.y.squeeze().numpy()
    test_x = test_data.x.numpy()
    test_y = test_data.y.squeeze().numpy()

    models = {
        "Logistic Regression":     LogisticRegression(),
        "Random Forest":           RandomForestClassifier(),
        "Gradient Boosting (GBM)": GradientBoostingClassifier(),
        "SVM":                     SVC(probability=True),
        "KNN":                     KNeighborsClassifier(),
    }
    results = {}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for name, model in models.items():
            model.fit(train_x, train_y)
            preds = model.predict_proba(test_x)[:, 1]
            auc = roc_auc_score(test_y, preds)
            results[name] = auc

    return results

In [ ]:
def regression_classfication_model_experience(train_data, test_data):
    r"""
    Train and evaluate multiple regression baseline models on time series data.

    Each model is trained on the training split and evaluated on the test split using
    Root Mean Squared Error (RMSE) as the performance metric.

    Args:
    - `train_data` (TimeSeriesDataset): data split for training
    - `test_data` (TimeSeriesDataset): data split for testing

    Return:
    - `results` (dict): dictionary mapping model name to its RMSE score on the test split
    """
    train_x = train_data.x.numpy()
    train_y = train_data.y.squeeze().numpy()
    test_x = test_data.x.numpy()
    test_y = test_data.y.squeeze().numpy()

    models = {
        "Linear Regression":       LinearRegression(),
        "Ridge":                   Ridge(alpha=1.0),
        "Lasso":                   Lasso(alpha=0.1),
        "Random Forest":           RandomForestRegressor(n_estimators=100, random_state=42),
        "Gradient Boosting (GBM)": GradientBoostingRegressor(n_estimators=100, random_state=42),
        "SVR":                     SVR(),
    }

    results = {}
    for name, model in models.items():
        model.fit(train_x, train_y)
        preds = model.predict(test_x)
        results[f"{name}"] = np.sqrt(mean_squared_error(test_y, preds))

    return results


In [ ]:
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               ExtraTreesClassifier, AdaBoostClassifier, BaggingClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import numpy as np
from sklearn.metrics import roc_auc_score
import warnings



def binary_classfication_model_experience(train_data, test_data):
    train_x = train_data.x.numpy()
    train_y = train_data.y.squeeze().numpy()
    test_x = test_data.x.numpy()
    test_y = test_data.y.squeeze().numpy()

    models = {
        # ── Baselines ────────────────────────────────────────────────
        "Majority Class (Baseline)":  DummyClassifier(strategy="most_frequent"),
        "Prior Probability (Baseline)": DummyClassifier(strategy="prior"),

        # ── Linear Models ────────────────────────────────────────────
        "Logistic Regression":        LogisticRegression(max_iter=1000),
        "LDA":                        LinearDiscriminantAnalysis(),
        "QDA":                        QuadraticDiscriminantAnalysis(),
        "Naive Bayes":                GaussianNB(),

        # ── Tree-based ───────────────────────────────────────────────
        "Random Forest":              RandomForestClassifier(),
        "Extra Trees":                ExtraTreesClassifier(),
        "Gradient Boosting (GBM)":    GradientBoostingClassifier(),
        "AdaBoost":                   AdaBoostClassifier(),
        "Bagging":                    BaggingClassifier(),

        # ── Modern Boosting (best for tabular/time series) ───────────
        "XGBoost":                    XGBClassifier(eval_metric="auc", verbosity=0),
        "LightGBM":                   LGBMClassifier(verbosity=-1),
        "CatBoost":                   CatBoostClassifier(verbose=0),

        # ── Distance / Kernel ────────────────────────────────────────
        "KNN":                        KNeighborsClassifier(),
        "SVM":                        SVC(probability=True),
    }

    # Models that don't support predict_proba natively
    no_proba_models = {"Ridge Classifier"}

    results = {}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for name, model in models.items():
            model.fit(train_x, train_y)

            if hasattr(model, "predict_proba"):
                preds = model.predict_proba(test_x)[:, 1]
            else:
                # Fallback: use decision function scores
                preds = model.decision_function(test_x)

            auc = roc_auc_score(test_y, preds)
            results[name] = auc

    return results

In [ ]:
def drop_feature_experiment(files : list, task: str, experiment_each_dataset: Callable = binary_classfication_model_experience):
    r"""
    Run a drop-feature experiment to evaluate the impact of removing each feature on model performance.

    For each feature in ALL_FEAT_COLUMN, all models are trained and evaluated with that feature
    removed. A baseline run with all features included is also performed. Results are averaged
    across all provided dataset files and summarized in a DataFrame.

    Args:
    - `files` (list): list of CSV file names to load datasets from
    - `task` (str): label column name used as the prediction target
    - `experiment_each_dataset` (callable): function to train and evaluate models on a single
      data split, defaults to `binary_classfication_model_experience`

    Return:
    - `result_df` (pd.DataFrame): DataFrame where each row corresponds to a feature removal setting
      (e.g. 'All features', 'W/o Close') and each column corresponds to a model's average
      performance score across all dataset files
    """
    data_loader = Dataloader(DATA_PATH)

    all_rows = []

    for feat in tqdm([None] + ALL_FEAT_COLUMN):
        all_results = {}
        used_feat = copy.deepcopy(ALL_FEAT_COLUMN)
        if feat != None:
            used_feat.remove(feat)
            print(f"INFO: Remove feature: {feat}")

        all_data_split = []
        for file_name in files:
            data = data_loader.from_csv(file_name, feat_columns=used_feat, label_column=task)
            train, val_test = data.split(VAL_START_DATE)
            val, test = val_test.split(TEST_START_DATE)
            all_data_split.append((train,val,test))
        
        feat_map = all_data_split[0][0].feat_map
        normalize_feat_exclude_idx = [idx for feat,idx in feat_map.items() if feat in ALREADY_NORMALIZE_FEAT]
        for train,val, test in all_data_split:
            normalize(train,val,test,exclude_cols = normalize_feat_exclude_idx)

        all_results = {}
        for train,val, test in all_data_split:
            result = experiment_each_dataset(train,test)
            for model, auc in result.items():
                if model not in all_results:
                    all_results[model] = []

                all_results[model].append(auc)
        average_result = {}
        average_result['Setting'] = 'All features' if feat is None else f'W/o {feat}'
        for model, all_auc in all_results.items():
            average_result[model] = sum(all_auc)/len(all_auc)

        all_rows.append(average_result)

    result_df = pd.DataFrame(all_rows)
    return result_df

In [ ]:
def feature_important_score_by_random_forest(files:list, task: str, model = RandomForestClassifier()):
    data_loader = Dataloader(DATA_PATH)
    all_data_split = []
    for file_name in files:
        data = data_loader.from_csv(file_name, feat_columns=ALL_FEAT_COLUMN, label_column=task)
        train, val_test = data.split(VAL_START_DATE)
        val, test = val_test.split(TEST_START_DATE)
        all_data_split.append((train,val,test))
    
    feat_map = all_data_split[0][0].feat_map
    normalize_feat_exclude_idx = [idx for feat,idx in feat_map.items() if feat in ALREADY_NORMALIZE_FEAT]
    all_importance = np.zeros(len(ALL_FEAT_COLUMN))
    for train,val, test in all_data_split:
        normalize(train,val,test,exclude_cols = normalize_feat_exclude_idx)

        train_x = train.x.numpy()
        train_y = train.y.squeeze().numpy()

        model.fit(train_x,train_y)
        all_importance = all_importance + np.array(model.feature_importances_)
    
    all_importance = all_importance / len(all_data_split)

    importance_df = pd.DataFrame({
        'feature': ALL_FEAT_COLUMN,
        'importance': all_importance
    }).sort_values('importance', ascending=False)
    return importance_df

    

## 2. Experiments and results

We conduct experiments on contribution of each feature on predicting `close price` of each token by removing each feature one by one and evaluate the performance of all baseline models on each token dataset for

1. Binary classification 
2. Regression 

### 2.1 Binary classification

Drop feature experiment

In [ ]:
files = os.listdir(DATA_PATH)
task = BINARY_LABEL_COLUMN

seed_everything(INIT_SEED)

result_df = drop_feature_experiment(files, task,binary_classfication_model_experience)

display(result_df)
result_df.to_csv("data/results/feature_selection_binary.csv")


NameError: name 'os' is not defined

Importance score from randomforest 


In [ ]:
files = os.listdir(DATA_PATH)
task = BINARY_LABEL_COLUMN

seed_everything(INIT_SEED)

result_df = feature_important_score_by_random_forest(files, task,RandomForestClassifier())

display(result_df)
result_df.to_csv("data/results/random_forest_feat_importance_binary.csv")

### 2.2 Regression

In [ ]:
files = os.listdir(DATA_PATH)
task = REGRESSION_LABEL_COLUMN

result_df = drop_feature_experiment(files, task,regression_classfication_model_experience)
result_df.to_csv("data/results/feature_selection_regression.csv")

display(result_df)

 25%|██▌       | 1/4 [00:05<00:17,  5.84s/it]

INFO: Remove feature: Close


 50%|█████     | 2/4 [00:11<00:11,  5.51s/it]

INFO: Remove feature: High


 75%|███████▌  | 3/4 [00:16<00:05,  5.38s/it]

INFO: Remove feature: Low


100%|██████████| 4/4 [00:21<00:00,  5.39s/it]


,Setting,Linear Regression,Ridge,Lasso,Random Forest,Gradient Boosting (GBM),SVR
0,All features,1.011674,1.081678,1.193678,1.683116,1.690810,8.105300
1,W/o Close,1.088909,1.146338,1.268263,1.759295,1.674885,8.111388
2,W/o High,1.015607,1.112159,1.193678,1.684193,1.724112,8.110004
3,W/o Low,1.005908,1.109144,1.193678,1.716842,1.652603,8.111540


In [ ]:
files = os.listdir(DATA_PATH)
task = REGRESSION_LABEL_COLUMN

seed_everything(INIT_SEED)

result_df = feature_important_score_by_random_forest(files, task,RandomForestRegressor())

display(result_df)
result_df.to_csv("data/results/random_forest_feat_importance_regression.csv")